In [12]:
!conda info --envs


# conda environments:
#
# * -> active
# + -> frozen
base                     C:\Users\simin\anaconda3
Gans                 *   C:\Users\simin\anaconda3\envs\Gans



# Gans – city pipeline (commented version)
Scrapes city information and population from Wikipedia and loads it into the MySQL database `gans`.

**Database tables**
- `cities` (static facts): `city_id`, `city_name`, `country`, `latitude`, `longitude`
- `populations` (changing facts, one row per run): `population_id`, `population`, `population_year`, `date_retrieved`, `city_id` → FK to `cities`

**Run order**
| Cell | What it does | Writes to MySQL? |
|---|---|---|
| 1 | imports, settings, password | no |
| 2 | scraping / extraction functions | no |
| 3 | pipeline functions | no |
| 4 | dry run (scrape only) | no |
| 5 | real run | **yes** |

**To add a city later:** run cell 5 with the new name, written exactly as at the end of its Wikipedia URL (spaces → `_`, e.g. `"New_York_City"`).

## 1. Imports and settings

In [1]:
# --- libraries ---
import re                          # regular expressions: find patterns in text (e.g. a 4-digit year)
import pandas as pd                # DataFrames + read_sql / to_sql
import requests                    # download web pages
from bs4 import BeautifulSoup      # turn HTML into a searchable "soup"
from datetime import date          # today's date for date_retrieved
import os                          # read settings from the operating system / .env
from dotenv import load_dotenv     # reads the .env file in this folder

# Wikipedia requires a User-Agent with a project name + contact, otherwise requests may be blocked
HEADERS = {'User-Agent': 'GansProject/1.0'}

load_dotenv()                                      # load secrets from .env (never stored in the notebook)
connection_string = os.getenv("CON_STRING")        # database address incl. password, from .env

## 2. Scraping and extraction functions
These functions only read data; they never touch MySQL.

In [2]:
def scrape_city(city):
    """Download the Wikipedia page of one city and return it as soup."""
    # build the link: base URL + city name as written in the Wikipedia URL
    url = "https://en.wikipedia.org/wiki/" + city
    response = requests.get(url, headers=HEADERS)            # download the page
    return BeautifulSoup(response.content, "html.parser")     # HTML -> soup


def extract_city_info(soup):
    """One row for the cities table: name, country, decimal coordinates."""
    # official page title (unique on Wikipedia, e.g. 'Frankfurt' vs 'Frankfurt (Oder)')
    city_name = soup.select("h1.firstHeading span.mw-page-title-main")[0].get_text(strip=True)

    # landmark "Country" (in a th) -> jump to the next td that holds the value
    # strip=True removes hidden spaces (we had ' Germany' for Hamburg once)
    # previous version: country = soup.find(string="Country").find_next("td").get_text(strip=True)
    country = soup.find("table", class_="infobox").find(string="Country").find_next("td").get_text(strip=True)

    # hidden decimal coordinates: <span class="geo">52.52000; 13.40500</span>
    coords = soup.find(class_="geo").get_text()
    parts = coords.split(";")                  # ['52.52000', ' 13.40500']
    latitude  = float(parts[0].strip())        # text -> decimal number
    longitude = float(parts[1].strip())

    # keys must match the MySQL column names exactly
    return {
        "city_name": city_name,
        "country":   country,
        "latitude":  latitude,
        "longitude": longitude
    }


def extract_population(soup):
    """One row for the populations table: name, population, year of the figure."""
    city_name = soup.select("h1.firstHeading span.mw-page-title-main")[0].get_text(strip=True)

    # search only inside the info box: "Population" can also appear in the
    # contents menu and in section headings (we found 3 matches on the Frankfurt page)
    infobox = soup.find("table", class_="infobox")                                    # ← NEW
    population_header = infobox.find(string="Population")                              # ← NEW

    # the number: check the next 5 td cells and take the first that LOOKS like a population
    population = None
    for td in population_header.find_all_next("td", limit=5):                           # ← CHANGED
        text = td.get_text(strip=True).split("[")[0]          # remove footnotes like [3]
        if re.fullmatch(r"\d{1,3}(,\d{3})+", text):            # digits with thousands commas
            population = int(text.replace(",", ""))
            break
    if population is None:
        raise ValueError("no population number found")

    # the year is in the same header, e.g. 'Population(2023)[3]'
    raw_header = population_header.find_parent("th").get_text(strip=True)              # ← CHANGED
    population_year = int(re.search(r"\d{4}", raw_header).group())

    return {
        "city_name": city_name,
        "population": population,
        "population_year": population_year
    }

## 3. Pipeline functions
Order matters: new cities must be in MySQL first, so they get a `city_id` before populations are loaded.

In [3]:
def scrape_cities(cities):
    """Download each city page ONCE; skip pages that cannot be parsed."""
    city_rows = []
    population_rows = []
    for city in cities:
        try:
            soup = scrape_city(city)              # one download per city
            info = extract_city_info(soup)        # extract both BEFORE appending,
            pop  = extract_population(soup)       # so a city is never added without its population
            city_rows.append(info)
            population_rows.append(pop)
        except Exception as error:
            # a page with a different layout is skipped instead of stopping the whole pipeline
            print(f"⚠️ Skipped {city}: {error}")
    # return two DataFrames at once -> catch them with two names
    return pd.DataFrame(city_rows), pd.DataFrame(population_rows)


def load_new_cities(cities_df):
    """Send only cities that are not in MySQL yet (avoids duplicates)."""
    cities_from_sql = pd.read_sql("cities", con=connection_string)        # what is already stored
    # ~ = NOT  ->  keep rows whose city_name is NOT already in the database
    new_cities_df = cities_df[~cities_df["city_name"].isin(cities_from_sql["city_name"])]

    if len(new_cities_df) > 0:
        # first argument = TABLE name; append keeps our table design (PK, AUTO_INCREMENT)
        new_cities_df.to_sql("cities", if_exists="append", con=connection_string, index=False)

    print(f"✅ {len(new_cities_df)} new cities added")


def load_populations(populations_df):
    """Look up city_id, add today's date, append to the populations table."""
    # read cities again AFTER load_new_cities -> newly added cities now have IDs too
    cities_from_sql = pd.read_sql("cities", con=connection_string)
    # swap city_name for city_id (same idea as author_name -> author_id in the books example)
    populations_df = populations_df.merge(cities_from_sql[["city_id", "city_name"]],
                                          on="city_name", how="left")
    # when WE collected the data (population_year = what year the figure describes)
    populations_df["date_retrieved"] = date.today()
    # keep exactly the MySQL columns (population_id is filled by MySQL)
    populations_df = populations_df[["population", "population_year", "date_retrieved", "city_id"]]
    # append = keep history: every run adds new rows with that day's date
    populations_df.to_sql("populations", if_exists="append", con=connection_string, index=False)
    print(f"✅ {len(populations_df)} population rows added")


def run_city_pipeline(cities):
    """Scrape the given cities and load them into MySQL (the only function you need to call)."""
    cities_df, populations_df = scrape_cities(cities)    # 1. extract + transform
    if cities_df.empty:                                  # nothing worked -> stop safely
        print("Nothing scraped. Check the city names.")
        return
    load_new_cities(cities_df)                           # 2. load cities (only new ones)
    load_populations(populations_df)                     # 3. load populations (always)

## 4. Dry run (scrapes only, does NOT write to MySQL)
Use this to test a new city before loading it. Check name, country, coordinates, population and year.

In [4]:
test_cities_df, test_populations_df = scrape_cities(["Berlin", "Hamburg", "Munich", "Frankfurt"])
display(test_cities_df)
display(test_populations_df)

,city_name,country,latitude,longitude
0,Berlin,Germany,52.52000,13.40500
1,Hamburg,Germany,53.55000,10.00000
2,Munich,Germany,48.13750,11.57500
3,Frankfurt,Germany,50.11056,8.68222


,city_name,population,population_year
0,Berlin,3596999,2022
1,Hamburg,1973896,2024
2,Munich,1505036,2025
3,Frankfurt,760656,2025


## 5. Run the pipeline (writes to MySQL)
⚠️ Remove the `#` to run. Each run appends a new population row per city (with today's date); new cities are added only once.

Check afterwards in MySQL Workbench:
```sql
SELECT c.city_name, p.population, p.population_year, p.date_retrieved
FROM populations p
JOIN cities c ON p.city_id = c.city_id;
```

In [11]:
# examples:
#run_city_pipeline(["Frankfurt"])                                   # add one new city
# run_city_pipeline(["Berlin", "Hamburg", "Munich", "Frankfurt"])    # refresh populations for all

✅ 1 new cities added
✅ 1 population rows added


In [5]:
pd.read_sql("cities", con=connection_string)

,city_id,city_name,country,latitude,longitude
0,1,Berlin,Germany,52.52000,13.40500
1,2,Hamburg,Germany,53.55000,10.00000
2,3,Munich,Germany,48.13750,11.57500
3,4,Frankfurt,Germany,50.11056,8.68222


## 5.1 Add new city in our city table

In [ ]:
#run once only and then add the # infront of it
#run_city_pipeline(["City_Name"])  

## 6. Checking git and gitignore

In [7]:
for f in sorted(os.listdir()):
    print(f)

.env
.gitignore
.ipynb_checkpoints
Gans_project_related_files
gans_pipeline_commented.ipynb
gans_weather.ipynb


In [9]:
gitignore_text = """# secrets
.env

# Jupyter auto-backups
.ipynb_checkpoints/

# course material and learning notebooks (not part of the project)
Gans_project_related_files/

# scratch notebooks
Untitled*.ipynb
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_text)

print(open(".gitignore").read())

# secrets
.env

# Jupyter auto-backups
.ipynb_checkpoints/

# course material and learning notebooks (not part of the project)
Gans_project_related_files/

# scratch notebooks
Untitled*.ipynb



In [10]:
from dotenv import dotenv_values

# collect the secrets from .env (values only, never printed)
env = dotenv_values(".env")
secrets = [env["WEATHER_KEY"]]
con = env["CON_STRING"]
secrets.append(con.split("root:")[1].split("@")[0])     # the password part of the connection string

# folders/files that will NOT be uploaded (from .gitignore)
skip = {".env", ".ipynb_checkpoints", "Gans_project_related_files", ".git"}

found = False
for folder, subfolders, files in os.walk("."):
    subfolders[:] = [s for s in subfolders if s not in skip]   # don't look inside skipped folders
    for name in files:
        if name in skip or name.startswith("Untitled"):
            continue
        path = os.path.join(folder, name)
        text = open(path, encoding="utf-8", errors="ignore").read()
        if any(s and s in text for s in secrets):
            print("⚠️ SECRET FOUND in:", path)
            found = True

print("✅ No secrets in files that will be uploaded." if not found else "❌ Fix the files above before using Git.")

✅ No secrets in files that will be uploaded.
